# MedVS-AI — Phase 0: Dermoscopy Segmentation Spike (Colab GPU)

Trains a U-Net to segment skin lesions (**ISIC 2018 Task 1**) on a free GPU and
checks it beats a trivial **Otsu baseline** — the gate that derisks the whole platform.

> **NOT FOR CLINICAL USE.** Research/educational only. No output is a diagnosis.

**Before you start:** `Runtime > Change runtime type > Hardware accelerator: T4 GPU`, then `Runtime > Run all`.
This notebook is self-contained — it mirrors the `ml/` code, so no repo clone is needed.


## 1. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU -> Runtime > Change runtime type > Hardware accelerator: T4 GPU")


## 2. Install the two extra packages
(Colab already ships a GPU build of torch + numpy — don't reinstall those.)

In [ ]:
%pip -q install onnxruntime onnxscript
print("done")


## 3. Config
Tune `MAX_IMAGES` / `EPOCHS` for a faster run.

In [ ]:
import torch
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 25                # from-scratch convergence is slower; raise for higher Dice
LR = 1e-3
VAL_FRACTION = 0.2
SEED = 42
MAX_IMAGES = None          # e.g. 800 for speed; None = all ~2594
THRESH_JACCARD_CUTOFF = 0.65
PRED_THRESHOLD = 0.5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGES_URL = "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task1-2_Training_Input.zip"
MASKS_URL  = "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task1_Training_GroundTruth.zip"
DATASET_LICENSE = "CC-BY-NC-4.0"
DATASET_ATTRIBUTION = "ISIC 2018 Challenge (Codella et al. 2019; Tschandl et al. 2018, HAM10000)"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 4. Download ISIC 2018 Task 1
Masks are small; the image zip is ~10 GB but downloads fast on Colab (a few minutes).

In [ ]:
import os, zipfile, urllib.request
IMAGES_DIR = "data/ISIC2018_Task1-2_Training_Input"
MASKS_DIR  = "data/ISIC2018_Task1_Training_GroundTruth"
os.makedirs("data", exist_ok=True)

def _hook(b, bs, total):
    if total > 0 and b % 500 == 0:
        print(f"  {min(100, b*bs*100//total)}%", end="\r")

def fetch(url, zname, outdir):
    if os.path.isdir(outdir) and os.listdir(outdir):
        print("already extracted:", outdir); return
    if not os.path.exists(zname):
        print("downloading", url)
        urllib.request.urlretrieve(url, zname, _hook)
    print("\nextracting", zname)
    with zipfile.ZipFile(zname) as z:
        z.extractall("data")

fetch(MASKS_URL, "data/masks.zip", MASKS_DIR)
fetch(IMAGES_URL, "data/images.zip", IMAGES_DIR)
print("images:", len(os.listdir(IMAGES_DIR)), "| masks:", len(os.listdir(MASKS_DIR)))


## 5. Build the manifest (records per-image license)

In [ ]:
import re, pandas as pd
ISIC_RE = re.compile(r"(ISIC_\d+)")
rows = []
for m in sorted(os.listdir(MASKS_DIR)):
    if not m.endswith(".png"):
        continue
    iid = ISIC_RE.search(m).group(1)
    ip = os.path.join(IMAGES_DIR, iid + ".jpg")
    if os.path.exists(ip):
        rows.append({"isic_id": iid, "image_path": ip,
                     "mask_path": os.path.join(MASKS_DIR, m),
                     "license": DATASET_LICENSE, "attribution": DATASET_ATTRIBUTION})
manifest = pd.DataFrame(rows)
if MAX_IMAGES:
    manifest = manifest.sample(n=min(MAX_IMAGES, len(manifest)), random_state=SEED).reset_index(drop=True)
print("image/mask pairs:", len(manifest))
manifest.head()


## 6. Dataset, split, metrics, overlay helpers

In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

def split_manifest(df, val_fraction=VAL_FRACTION, seed=SEED):
    s = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = int(len(s) * val_fraction)
    return s.iloc[n:].reset_index(drop=True), s.iloc[:n].reset_index(drop=True)

class ISICDataset(Dataset):
    def __init__(self, df, size=IMG_SIZE, augment=False):
        self.df = df.reset_index(drop=True); self.size = size; self.augment = augment
        self.mean = np.array(IMAGENET_MEAN, np.float32); self.std = np.array(IMAGENET_STD, np.float32)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r["image_path"]).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        msk = Image.open(r["mask_path"]).convert("L").resize((self.size, self.size), Image.NEAREST)
        x = np.asarray(img, np.float32) / 255.0
        y = (np.asarray(msk, np.float32) > 127).astype(np.float32)
        if self.augment and np.random.rand() < 0.5:
            x = x[:, ::-1, :].copy(); y = y[:, ::-1].copy()
        x = (x - self.mean) / self.std
        return torch.from_numpy(x.transpose(2, 0, 1)), torch.from_numpy(y[None])

def raw_pair(r, size=IMG_SIZE):
    img = np.asarray(Image.open(r["image_path"]).convert("RGB").resize((size, size), Image.BILINEAR), np.uint8)
    gt = (np.asarray(Image.open(r["mask_path"]).convert("L").resize((size, size), Image.NEAREST)) > 127).astype(np.uint8)
    return img, gt

def _b(a, t=0.5): return (np.asarray(a) >= t).astype(np.uint8)
def dice_coef(p, g, t=0.5):
    p, g = _b(p, t), _b(g, 0.5); ps, gs = p.sum(), g.sum()
    if ps == 0 and gs == 0: return 1.0
    return float(2 * np.logical_and(p, g).sum() / (ps + gs))
def iou(p, g, t=0.5):
    p, g = _b(p, t), _b(g, 0.5); u = np.logical_or(p, g).sum()
    return 1.0 if u == 0 else float(np.logical_and(p, g).sum() / u)
def threshold_jaccard(p, g, c=THRESH_JACCARD_CUTOFF, t=0.5):
    j = iou(p, g, t); return j if j >= c else 0.0
def aggregate(preds, gts):
    d = [dice_coef(p, g) for p, g in zip(preds, gts)]
    i = [iou(p, g) for p, g in zip(preds, gts)]
    tj = [threshold_jaccard(p, g) for p, g in zip(preds, gts)]
    n = max(len(d), 1)
    return {"n": len(d), "dice": sum(d)/n, "iou": sum(i)/n, "threshold_jaccard": sum(tj)/n}

def overlay(img, mask, color=(255, 0, 0), alpha=0.4):
    out = img.astype(np.float32).copy(); m = mask.astype(bool)
    for c in range(3):
        out[..., c][m] = (1 - alpha) * out[..., c][m] + alpha * color[c]
    return np.clip(out, 0, 255).astype(np.uint8)

train_df, val_df = split_manifest(manifest)
print("train:", len(train_df), "| val:", len(val_df))


## 7. Otsu baseline (the bar to beat)

In [ ]:
def otsu_threshold(gray):
    hist, _ = np.histogram(gray, bins=256, range=(0, 256)); total = gray.size
    sumtot = np.dot(np.arange(256), hist); sumb = wb = 0.0; best = -1.0; bt = 0
    for t in range(256):
        wb += hist[t]
        if wb == 0: continue
        wf = total - wb
        if wf == 0: break
        sumb += t * hist[t]; mb = sumb / wb; mf = (sumtot - sumb) / wf
        v = wb * wf * (mb - mf) ** 2
        if v > best: best = v; bt = t
    return bt

def otsu_mask(rgb):
    g = rgb.mean(2).astype(np.uint8)
    return (g <= otsu_threshold(g)).astype(np.uint8)

bpred, bgt = [], []
for _, r in val_df.iterrows():
    img, gt = raw_pair(r); bpred.append(otsu_mask(img)); bgt.append(gt)
baseline = aggregate(bpred, bgt); baseline["method"] = "otsu"
print("BASELINE:", baseline)


## The model — Attention U-Net, built from scratch
A U-Net assembled by hand from `torch.nn` primitives (`Conv2d` / `BatchNorm2d` / `ReLU` / `MaxPool2d` / `ConvTranspose2d`) with **additive attention gates** (Oktay et al., 2018) on every skip connection — the gate lets the decoder down-weight irrelevant encoder features and focus on the lesion. No `segmentation_models_pytorch`; the Dice + BCE loss is hand-written too.

> Trained **from scratch** (no ImageNet pretraining), so it needs more epochs than a transfer-learned encoder and may land a little below it — that's the honest cost of building it yourself. Bump `EPOCHS` or `base` for more capacity.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class DoubleConv(nn.Module):
    """(3x3 conv -> BatchNorm -> ReLU) x2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class AttentionGate(nn.Module):
    """Additive attention gate (Oktay et al. 2018): reweight the encoder skip `x`
    by an attention map computed from `x` and the decoder gating signal `g`, so the
    decoder can suppress irrelevant background and focus on the lesion."""
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.theta_x = nn.Conv2d(x_ch, inter_ch, 1, bias=False)
        self.phi_g   = nn.Conv2d(g_ch, inter_ch, 1, bias=True)
        self.psi     = nn.Conv2d(inter_ch, 1, 1, bias=True)
    def forward(self, g, x):
        alpha = torch.sigmoid(self.psi(F.relu(self.theta_x(x) + self.phi_g(g))))
        return x * alpha

class UpBlock(nn.Module):
    """Upsample -> attention-gate the skip -> concat -> DoubleConv."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.att  = AttentionGate(g_ch=out_ch, x_ch=skip_ch, inter_ch=out_ch // 2)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        skip = self.att(x, skip)
        return self.conv(torch.cat([skip, x], dim=1))

class AttentionUNet(nn.Module):
    """U-Net hand-built from conv primitives, with an attention gate on every skip."""
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        c = [base, base*2, base*4, base*8, base*16]
        self.e1, self.e2 = DoubleConv(in_ch, c[0]), DoubleConv(c[0], c[1])
        self.e3, self.e4 = DoubleConv(c[1], c[2]), DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[4])
        self.u4, self.u3 = UpBlock(c[4], c[3], c[3]), UpBlock(c[3], c[2], c[2])
        self.u2, self.u1 = UpBlock(c[2], c[1], c[1]), UpBlock(c[1], c[0], c[0])
        self.head = nn.Conv2d(c[0], out_ch, 1)
    def forward(self, x):
        s1 = self.e1(x); s2 = self.e2(self.pool(s1))
        s3 = self.e3(self.pool(s2)); s4 = self.e4(self.pool(s3))
        b = self.bottleneck(self.pool(s4))
        x = self.u4(b, s4); x = self.u3(x, s3); x = self.u2(x, s2); x = self.u1(x, s1)
        return self.head(x)           # raw logits [B, 1, H, W]

def dice_bce_loss(logits, target, eps=1.0):
    """Dice + BCE, both hand-written (no smp)."""
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    num = 2 * (p * target).sum((1, 2, 3)) + eps
    den = (p + target).sum((1, 2, 3)) + eps
    return bce + (1 - (num / den).mean())

print("AttentionUNet params (base=32): %.2fM" % (sum(p.numel() for p in AttentionUNet().parameters()) / 1e6))


## 8. Train the U-Net

In [ ]:
torch.manual_seed(SEED); np.random.seed(SEED)
model = AttentionUNet(in_ch=3, out_ch=1, base=32).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

train_dl = DataLoader(ISICDataset(train_df, augment=True), batch_size=BATCH_SIZE,
                      shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(ISICDataset(val_df), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=2, pin_memory=True)

def val_dice():
    model.eval(); s = []
    with torch.no_grad():
        for x, y in val_dl:
            p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
            for pi, gi in zip(p, y.numpy()):
                s.append(dice_coef(pi[0], gi[0]))
    return float(np.mean(s))

hist = {"loss": [], "val_dice": []}; best = -1.0
for e in range(1, EPOCHS + 1):
    model.train(); ls = []
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = dice_bce_loss(model(x), y)
        loss.backward(); opt.step(); ls.append(loss.item())
    sched.step(); vd = val_dice()
    hist["loss"].append(float(np.mean(ls))); hist["val_dice"].append(vd)
    tag = ""
    if vd > best:
        best = vd; torch.save(model.state_dict(), "unet.pt"); tag = " *saved"
    print(f"epoch {e:2d}/{EPOCHS}  loss={np.mean(ls):.4f}  val_dice={vd:.4f}{tag}")
print("best val Dice:", round(best, 4))


## 9. Training curve

In [ ]:
import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(hist["loss"], "C0-"); ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="C0")
ax2 = ax1.twinx(); ax2.plot(hist["val_dice"], "C1-"); ax2.set_ylabel("val Dice", color="C1")
plt.title("U-Net training"); plt.show()


## 10. Evaluate vs baseline — **THE GATE**

In [ ]:
model.load_state_dict(torch.load("unet.pt", map_location=DEVICE)); model.eval()
preds, gts = [], []
vds = ISICDataset(val_df)
with torch.no_grad():
    for i in range(len(vds)):
        x, y = vds[i]
        p = torch.sigmoid(model(x[None].to(DEVICE)))[0, 0].cpu().numpy()
        preds.append((p >= PRED_THRESHOLD).astype(np.uint8))
        gts.append(y[0].numpy().astype(np.uint8))
model_m = aggregate(preds, gts); model_m["method"] = "unet"
gate = model_m["dice"] > baseline["dice"]
print("MODEL   :", model_m)
print("BASELINE:", baseline)
print(f"\nDice improvement over baseline: {model_m['dice'] - baseline['dice']:+.4f}")
print("GATE PASSED \u2713" if gate else "GATE FAILED \u2717 - model does not beat Otsu")


## 11. Sanity overlays (image | ground truth | model)

In [ ]:
n = min(6, len(val_df))
fig, ax = plt.subplots(n, 3, figsize=(9, 3 * n))
for i in range(n):
    img, gt = raw_pair(val_df.iloc[i])
    ax[i, 0].imshow(img); ax[i, 0].set_title("image")
    ax[i, 1].imshow(overlay(img, gt, (0, 255, 0))); ax[i, 1].set_title("ground truth")
    ax[i, 2].imshow(overlay(img, preds[i], (255, 0, 0))); ax[i, 2].set_title("model")
    for a in ax[i]: a.axis("off")
plt.tight_layout(); plt.show()


## 12. Export ONNX + verify CPU inference

In [ ]:
model.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(model, dummy, "unet.onnx", input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
                  opset_version=17, dynamo=False)
import onnxruntime as ort
sess = ort.InferenceSession("unet.onnx", providers=["CPUExecutionProvider"])
with torch.no_grad(): to = model(dummy).numpy()
oo = sess.run(None, {"input": dummy.numpy()})[0]
print("max |torch - onnx| =", float(np.abs(to - oo).max()), "(want < 1e-3)")
print("ONNX CPU inference verified ✓")


## 13. Download the model

Bring `unet.onnx` back into your repo at `ml/models/unet.onnx` — that's the
artifact Phase 1's web backend will serve.

In [ ]:
from google.colab import files
files.download("unet.onnx")
# files.download("unet.pt")   # uncomment for the PyTorch checkpoint too
